In [12]:
import warnings

warnings.filterwarnings('ignore')

In [20]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
# from src.altering_plate_MAD import *
# from src.bridging_coefficients import *
# from src.distribution_assessment import *
# from src.adat_handling import read_adat_file
from src.dataframe_transformation import *
# from src.prophet_scores import * 
from modeling import rap_response_predictor
from __future__ import annotations


1. gather the data

In [7]:
# # get base RFUs from Izhar's parquet files
# base_rfus = pd.read_parquet('../../data/adat/base/v_166_OH2026_004.parquet').reset_index(drop=False)
# info_df = pd.read_csv('../../data/excels/20260126_adat_data_samples_with_sample_info.csv')
base_rfus_path = '../../data/adat/base/v_166_OH2026_004.parquet'
info_df_path = '../../data/excels/20260126_adat_data_samples_with_sample_info.csv'


In [8]:
def getting_base_rfu_data(base_rfus_path, info_path, ds_plates=[], nds_plates=[]):
    
    
    #Reading the data
    base_rfus = pd.read_parquet(base_rfus_path).reset_index(drop=False)
    info_df = pd.read_csv(info_path)

    #Cleaning PlateId values
    base_rfus['PlateId'] = base_rfus['PlateId'].replace('OH2023_001', 'OH2023_01')
    info_df['PlateId'] = info_df['PlateId'].str.split('_set').str[0]
    base_rfus['PlateId'] = base_rfus['PlateId'].str.split('_set').str[0]
    base_rfus['PlateId'] = base_rfus['PlateId'].str.split('_Lot3').str[0]
    base_rfus = add_measure_id(base_rfus)
    
    #Filtering
    edta_info_df = filter_dataframe(info_df, info_df['SampleMatrixType']=='EDTA Plasma',
                                info_df['SubType']=='PROPHET')
    edta_rfu_df = base_rfus[base_rfus['MeasureId'].isin(edta_info_df['MeasureId'])].drop_duplicates(subset=['MeasureId']).reset_index(drop=True)
    
    ds_info_df = filter_dataframe(info_df, info_df['SampleMatrixType']=='Streck',
                                info_df['SubType']=='PROPHET')
    ds_rfu_df = base_rfus[base_rfus['MeasureId'].isin(ds_info_df['MeasureId'])].drop_duplicates(subset=['MeasureId']).reset_index(drop=True)

    nds_info_df = filter_dataframe(info_df, info_df['SampleMatrixType']=='non-DS Streck')
    nds_rfu_df = base_rfus[base_rfus['MeasureId'].isin(nds_info_df['MeasureId'])].drop_duplicates(subset=['MeasureId']).reset_index(drop=True)




    return base_rfus, info_df, edta_rfu_df, ds_rfu_df, nds_rfu_df

In [13]:
base_rfus, info_df, edta_rfu_df, ds_rfu_df, nds_rfu_df = getting_base_rfu_data(base_rfus_path, info_df_path)

Added MeasureId


In [14]:
edta_rfu_df.shape

(1238, 7626)

In [31]:
plates

['OH2025_051',
 'OH2025_052',
 'OH2025_053',
 'OH2025_054',
 'OH2025_055',
 'OH2025_056',
 'OH2025_057',
 'OH2026_001',
 'OH2026_002',
 'OH2026_003']

In [32]:
plates = [f'OH2025_05{i}' for i in range(1, 8)]
plates += [f'OH2026_00{i}' for i in range(1, 4)]


In [36]:
ds_rfu_df = ds_rfu_df[ds_rfu_df['PlateId'].isin(plates)]
nds_rfu_df = nds_rfu_df[nds_rfu_df['PlateId'].isin(plates)]

In [37]:
nds_rfu_df.shape

(75, 7626)

2. create the transformation dicts

In [38]:
aptamers = [col for col in edta_rfu_df.columns if col[0].isdigit()]

In [39]:

def build_median_mad_transform_dict(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    *,
    aptamer_cols: list[str] | None = None,
    dropna: bool = True,
    mad_scale: str = "raw",  # "raw" (median absolute deviation) or "scaled" (1.4826*MAD)
    min_non_na: int = 3,
) -> dict[str, dict[str, float]]:
    """
    Build per-aptamer transform parameters to map SOURCE -> TARGET distribution
    using robust location (median) and scale (MAD).

    Returns:
      {
        "<aptamer>": {
            "source_median": ...,
            "source_MAD": ...,
            "target_median": ...,
            "target_MAD": ...
        },
        ...
      }

    Notes:
    - Expects source_df and target_df to contain the same aptamer columns (wide format).
    - MAD here means median(|x - median(x)|).
    - mad_scale:
        "raw"    -> MAD as defined above
        "scaled" -> 1.4826 * MAD (consistent with normal-distribution sigma estimate)
    - Columns with insufficient data (min_non_na) are skipped.
    """
    if aptamer_cols is None:
        aptamer_cols = sorted(set(source_df.columns).intersection(set(target_df.columns)))
    else:
        missing_s = [c for c in aptamer_cols if c not in source_df.columns]
        missing_t = [c for c in aptamer_cols if c not in target_df.columns]
        if missing_s or missing_t:
            raise ValueError(
                f"Missing columns. "
                f"source missing: {missing_s[:10]}{'...' if len(missing_s) > 10 else ''}; "
                f"target missing: {missing_t[:10]}{'...' if len(missing_t) > 10 else ''}"
            )

    if mad_scale not in ("raw", "scaled"):
        raise ValueError("mad_scale must be 'raw' or 'scaled'.")

    scale_factor = 1.4826 if mad_scale == "scaled" else 1.0

    def median_and_mad(s: pd.Series) -> tuple[float, float] | None:
        x = pd.to_numeric(s, errors="coerce")
        if dropna:
            x = x.dropna()
        if len(x) < min_non_na:
            return None
        med = float(np.nanmedian(x.to_numpy()))
        mad = float(np.nanmedian(np.abs(x.to_numpy() - med))) * scale_factor
        return med, mad

    out: dict[str, dict[str, float]] = {}

    for apt in aptamer_cols:
        src = median_and_mad(source_df[apt])
        tgt = median_and_mad(target_df[apt])
        if src is None or tgt is None:
            continue

        src_med, src_mad = src
        tgt_med, tgt_mad = tgt

        out[apt] = {
            "source_median": src_med,
            "source_MAD": src_mad,
            "target_median": tgt_med,
            "target_MAD": tgt_mad,
        }

    return out

In [40]:
ds_params = build_median_mad_transform_dict(source_df=ds_rfu_df, target_df=edta_rfu_df, aptamer_cols=aptamers)
nds_params = build_median_mad_transform_dict(source_df=nds_rfu_df, target_df=edta_rfu_df, aptamer_cols=aptamers)

In [41]:
ds_rfu_df['10000-28'].median()

np.float64(1332.75)

In [42]:
import json
with open('../../data/params/pre_hybridization/prehybridization_ds_params.json', 'w') as f:
    json.dump(ds_params, f)
with open('../../data/params/pre_hybridization/prehybridization_nds_params.json', 'w') as f:
    json.dump(nds_params, f)

3. Transform text files

In [25]:
from code_script import *

In [26]:
os.getcwd()

'c:\\Users\\RoeeOrland\\OneDrive - Oncohost DX\\Playground - Hub\\Roee\\bridging\\projects\\pre_post_bridging'

In [49]:

for plate in plates[:1]:
    start = time.time()
        # plate = f"OH2026_00{i}"
        
# i = 2
# if i == 3:
        # if i > -5:
    alter_scanner_files_MAD_bridging(f'../../data/plates/original_with_altered_workbooks/{plate}',
                f'../../data/plates/original_with_altered_workbooks/{plate}/{plate} Workbook.xlsx',
                
                {'Streck': '../../data/params/pre_hybridization/ds_11022026.json',
                'NDS': '../../data/params/pre_hybridization/nds_11022026.json'},
                f'results/plates/madmed_transformed_03032026/{plate}',
                
    )
            # alter_scanner_files_MAD_bridging(f'data/plates/multiple_types_experiment/OH2025_05{i}_altered_workbook',
            #                     f'data/plates/multiple_types_experiment/OH2025_05{i}_altered_workbook/OH2025_05{i} Workbook.xlsx',
            #                     {'Streck': 'data/conversion_coefficients/MAD_median_streck_coefficients.json',
            #                     'nds': 'data/conversion_coefficients/nds_params.json'},
            #                     f'data/plates/multiple_types_experiment/results/testing_transformation/OH2025_05{i}_for testing',
                            
            # )
    print(f"Time for plate {plate}: {time.time() - start} seconds")


Tab 4 - Plate Map
Tab 7 - Sample Import Sheet
Tab 4 - Plate Map
Tab 7 - Sample Import Sheet
Altered: SG21378706_258736525643_S001_GE1_1200_Jun14Enhanced7Rep_1_4.txt
Altered: SG21378706_258736525643_S001_GE1_1200_Jun14Enhanced7Rep_2_3.txt
Altered: SG21378706_258736525644_S001_GE1_1200_Jun14Enhanced7Rep_1_1.txt
Altered: SG21378706_258736525644_S001_GE1_1200_Jun14Enhanced7Rep_2_1.txt
Altered: SG21378706_258736525645_S001_GE1_1200_Jun14Enhanced7Rep_2_2.txt
Altered: SG21378706_258736525646_S001_GE1_1200_Jun14Enhanced7Rep_2_3.txt
Altered: SG21378706_258736525647_S001_GE1_1200_Jun14Enhanced7Rep_1_1.txt
Altered: SG21378706_258736525647_S001_GE1_1200_Jun14Enhanced7Rep_2_1.txt
Altered: SG21378706_258736525648_S001_GE1_1200_Jun14Enhanced7Rep_1_4.txt
Done!
Time for plate OH2025_051: 34.51544737815857 seconds


4. transform ADATs

In [1]:
import os

In [2]:
os.getcwd()

'c:\\Users\\RoeeOrland\\OneDrive - Oncohost DX\\Playground - Hub\\Roee\\bridging\\projects\\pre_post_bridging'

In [6]:
from src.adat_handling import read_adat_file

In [10]:
import pandas as pd
from pathlib import Path

folder = Path("results/adats/")

dfs = []

for file in folder.rglob("*.adat"):  # search recursively in subfolders
    if "anmlSMP.adat" in file.name:
        df, _ = read_adat_file(str(file))
        dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

c:\Users\RoeeOrland\OneDrive - Oncohost DX\Playground - Hub\Roee\bridging\bridging_env\Lib\site-packages\somadata\io\adat\file.py:149: UserWarning: Adding empty values to column metadata: "SeqIdVersion"
  warnings.warn(f'Adding empty values to column metadata: "{name}"')
c:\Users\RoeeOrland\OneDrive - Oncohost DX\Playground - Hub\Roee\bridging\bridging_env\Lib\site-packages\somadata\io\adat\file.py:149: UserWarning: Adding empty values to column metadata: "SeqIdVersion"
  warnings.warn(f'Adding empty values to column metadata: "{name}"')
c:\Users\RoeeOrland\OneDrive - Oncohost DX\Playground - Hub\Roee\bridging\bridging_env\Lib\site-packages\somadata\io\adat\file.py:149: UserWarning: Adding empty values to column metadata: "SeqIdVersion"
  warnings.warn(f'Adding empty values to column metadata: "{name}"')
c:\Users\RoeeOrland\OneDrive - Oncohost DX\Playground - Hub\Roee\bridging\bridging_env\Lib\site-packages\somadata\io\adat\file.py:149: UserWarning: Adding empty values to column metada

In [11]:
combined_df.shape

(528, 7633)

In [14]:
from src.prophet_scores import get_prophet_score
from modeling.rap_response_predictor import *

In [15]:
model = pd.read_pickle('../../data/models/DCB_nosex_gamma1_rescaled.pkl')

In [16]:
combined_df['PROphetScore'] = get_prophet_score(combined_df, model)

AssertionError: test_df contains NaN values. NaNs are not supported.

In [23]:
for i in range(10):
    get_prophet_score(combined_df.iloc[i:i+1, :], model)

KeyboardInterrupt: 

,PlateId,PlateRunDate,ScannerID,PlatePosition,SlideId,Subarray,SampleId,SampleType,PercentDilution,SampleMatrix,...,9984-12,9986-14,9987-30,9989-12,9991-112,9993-11,9994-217,9995-6,9997-12,9999-1
1,OH2025_051,2025-11-19,SG21378706,A2,258736525644,1,463434142,Sample,20,Streck,...,720.7,13456.3,558.6,383.6,612.9,1013.8,2200.5,2747.1,6519.8,156.0


In [24]:
get_prophet_score(combined_df.iloc[i:i+1, :], model)

KeyboardInterrupt: 

5. test PROphet scores